In [30]:
import random
audio1 = [random.randint(-20, 20) for _ in range(1000000)]
audio2 = [random.randint(-20, 20) for _ in range(1000000)]

# Online Source 

In [31]:
#https://www.pinecone.io/learn/series/faiss/locality-sensitive-hashing/

In [32]:
print(audio1)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [47]:
# Function to generate shingles
def create_shingles(audio, k ):
    shingles = set()
    for i in range(len(audio) - k + 1):
        shingle = tuple(audio[i:i+k])
        shingles.add(shingle)
    return shingles

In [48]:
#creating shingles
shingles1 = create_shingles(audio1,1)
shingles2 = create_shingles(audio2,1)

In [49]:
shingles2

{(-20,),
 (-19,),
 (-18,),
 (-17,),
 (-16,),
 (-15,),
 (-14,),
 (-13,),
 (-12,),
 (-11,),
 (-10,),
 (-9,),
 (-8,),
 (-7,),
 (-6,),
 (-5,),
 (-4,),
 (-3,),
 (-2,),
 (-1,),
 (0,),
 (1,),
 (2,),
 (3,),
 (4,),
 (5,),
 (6,),
 (7,),
 (8,),
 (9,),
 (10,),
 (11,),
 (12,),
 (13,),
 (14,),
 (15,),
 (16,),
 (17,),
 (18,),
 (19,),
 (20,)}

In [50]:
shingles1

{(-20,),
 (-19,),
 (-18,),
 (-17,),
 (-16,),
 (-15,),
 (-14,),
 (-13,),
 (-12,),
 (-11,),
 (-10,),
 (-9,),
 (-8,),
 (-7,),
 (-6,),
 (-5,),
 (-4,),
 (-3,),
 (-2,),
 (-1,),
 (0,),
 (1,),
 (2,),
 (3,),
 (4,),
 (5,),
 (6,),
 (7,),
 (8,),
 (9,),
 (10,),
 (11,),
 (12,),
 (13,),
 (14,),
 (15,),
 (16,),
 (17,),
 (18,),
 (19,),
 (20,)}

# creating vocabulary by taking union

In [36]:
vocabulary = sorted(set.union(shingles1,shingles2))

In [37]:
vocabulary

[(-20,),
 (-19,),
 (-18,),
 (-17,),
 (-16,),
 (-15,),
 (-14,),
 (-13,),
 (-12,),
 (-11,),
 (-10,),
 (-9,),
 (-8,),
 (-7,),
 (-6,),
 (-5,),
 (-4,),
 (-3,),
 (-2,),
 (-1,),
 (0,),
 (1,),
 (2,),
 (3,),
 (4,),
 (5,),
 (6,),
 (7,),
 (8,),
 (9,),
 (10,),
 (11,),
 (12,),
 (13,),
 (14,),
 (15,),
 (16,),
 (17,),
 (18,),
 (19,),
 (20,)]

# Now doing one hot encoding

In [38]:
def one_hot_encoding(shingles, vocabulary):
    encoding = [1 if shingle in shingles else 0 for shingle in vocabulary]
    return encoding

In [39]:
encoding1 = one_hot_encoding(shingles1,vocabulary)

In [40]:
encoding2 = one_hot_encoding(shingles2,vocabulary)

In [45]:
encoding1

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [44]:
hash_ex = list(range(1, len(encoding1)+1))
print(hash_ex)  # we haven't shuffled yet

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]


In [51]:
from random import shuffle

shuffle(hash_ex)
print(hash_ex)

[30, 3, 38, 11, 35, 41, 4, 37, 22, 18, 8, 34, 21, 28, 33, 1, 12, 14, 16, 25, 7, 9, 6, 39, 24, 32, 15, 27, 10, 23, 2, 29, 31, 26, 13, 19, 20, 40, 5, 17, 36]


In [53]:
print(f"7 -> {hash_ex.index(7)}") # note that value 7 can be found at index 1 in hash_ex

7 -> 20


In [65]:
for i in range(1, len(vocabulary)+1):
    idx = hash_ex.index(i)
    signature_val = encoding1[idx]
    print(f"{i} -> {idx} -> {signature_val}")
    if signature_val == 1:
        print('match!')
        break

1 -> 15 -> 1
match!


In [66]:
def create_hash_func(size: int):
    # function for creating the hash vector/function
    hash_ex = list(range(1, len(vocabulary)+1))
    shuffle(hash_ex)
    return hash_ex

def build_minhash_func(vocab_size: int, nbits: int):
    # function for building multiple minhash vectors
    hashes = []
    for _ in range(nbits):
        hashes.append(create_hash_func(vocab_size))
    return hashes

# we create 20 minhash vectors
minhash_func = build_minhash_func(len(vocabulary), 20)

In [75]:
minhash_func

[[10,
  29,
  21,
  13,
  34,
  20,
  8,
  12,
  40,
  14,
  17,
  22,
  1,
  27,
  4,
  33,
  16,
  15,
  24,
  2,
  35,
  23,
  28,
  36,
  9,
  30,
  25,
  38,
  18,
  6,
  41,
  5,
  37,
  32,
  31,
  11,
  26,
  19,
  7,
  39,
  3],
 [8,
  36,
  39,
  23,
  11,
  16,
  22,
  40,
  14,
  26,
  38,
  30,
  6,
  35,
  19,
  13,
  25,
  18,
  29,
  3,
  33,
  24,
  31,
  2,
  20,
  41,
  9,
  27,
  5,
  21,
  1,
  34,
  32,
  10,
  28,
  17,
  37,
  4,
  7,
  15,
  12],
 [39,
  13,
  7,
  1,
  29,
  20,
  23,
  25,
  14,
  2,
  17,
  18,
  3,
  21,
  10,
  22,
  38,
  35,
  6,
  12,
  34,
  36,
  40,
  15,
  11,
  37,
  31,
  33,
  4,
  5,
  26,
  24,
  41,
  16,
  9,
  27,
  32,
  8,
  28,
  30,
  19],
 [40,
  15,
  6,
  12,
  34,
  29,
  24,
  38,
  3,
  36,
  18,
  4,
  39,
  30,
  28,
  16,
  21,
  8,
  35,
  14,
  27,
  41,
  26,
  37,
  31,
  17,
  19,
  7,
  23,
  33,
  25,
  9,
  13,
  20,
  5,
  32,
  10,
  22,
  11,
  2,
  1],
 [15,
  32,
  35,
  28,
  8,
  38,
  5,
  11,
  

In [73]:
def create_hash(vector: list):
    # use this function for creating our signatures (eg the matching)
    signature = []
    for func in minhash_func:
        for i in range(1, len(vocabulary)+1):
            idx = func.index(i)
            signature_val = vector[idx]
            if signature_val in vocabulary and signature_val % 5 == 0:
                signature.append(idx)
                break
    return signature

In [74]:
# now create signatures
a_sig = create_hash(encoding1)
b_sig = create_hash(encoding2)

print(a_sig)
print(b_sig)   #maybe It's because both rand generated are same or maybe it's the condition that signature must be 1%5

[]
[]


# Now for similarity 'DICE'

In [77]:
#Dice(A,B)=2 x (length (A intersection B)/sum (Length(A)+Length (B)))
def DICE(set1,set2):
    intersection = len(set1.intersection(set2))
    lenA=len(set1)
    lenB=len(set2)
    return 2*(intersection)/(lenA+lenB)

In [79]:
print("DICE = ")
DICE(set(shingles1),set(shingles2))

DICE = 


1.0